In [11]:
# !pip install sentence_transformers
#!pip install chromadb

   ---------------------------------------- 0.0/740.6 kB ? eta -:--:--
   ---------------------------------------- 740.6/740.6 kB 6.0 MB/s  0:00:00
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ------ --------------------------------- 1.3/8.2 MB 7.0 MB/s eta 0:00:01
   -------------- ------------------------- 2.9/8.2 MB 6.7 MB/s eta 0:00:01
   ------------------- -------------------- 3.9/8.2 MB 6.5 MB/s eta 0:00:01
   -------------------------- ------------- 5.5/8.2 MB 6.6 MB/s eta 0:00:01
   --------------------------------- ------ 6.8/8.2 MB 6.4 MB/s eta 0:00:01
   ---------------------------------------  8.1/8.2 MB 6.6 MB/s eta 0:00:01
   ---------------------------------------- 8.2/8.2 MB 6.5 MB/s  0:00:01

   ---------------- ----------------------- 2/5 [joblib]
   ---------------- ----------------------- 2/5 [joblib]
   ---------------- ----------------------- 2/5 [joblib]
   ------------------------ --------------- 3/5 [scikit-learn]
   ----------------

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Any

import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

In [2]:
POLICY_DOCS = {
    "gold_card_fee": (
        "Gold Credit Card: Annual Fee is $50. The annual fee is waived if annual "
        "card spending exceeds $10,000. Interest rate is 18% APR."
    ),
    "student_account": (
        "Student Account: Zero minimum balance is required. There are no monthly "
        "maintenance fees. Maximum daily cash withdrawal is $500."
    ),
    "loan_eligibility": (
        "Personal Loan: Applicants generally require a credit score above 700 and "
        "annual income above $50,000. Maximum loan tenure is five years."
    ),
    "kyc_policy": (
        "KYC policy: A valid government-issued ID, such as a passport or driver's "
        "license, is required. Address proof is required for standard accounts."
    ),
    "fraud_policy": (
        "Fraud policy: Suspected unauthorized transactions must be escalated to a "
        "human fraud-support specialist. The assistant must not investigate or modify "
        "the account directly."
    ),
}


In [3]:
PROJECT_ROOT = r"C:\Users\HP\OneDrive\emeritius\Capstone Project\Agentic_AI_Capstone_Modular"

In [4]:
class BankingRAG:
    """Persistent ChromaDB collection that returns semantically relevant policy text."""

    def __init__(
        self,
        persist_directory: str | Path | None = None,
        collection_name: str = "banking_policies",
        embedding_model: str = "all-MiniLM-L6-v2",
    ) -> None:
        # project_root = Path(__file__).resolve()
        db_path = (
            Path(persist_directory)
            if persist_directory
            else project_root / "memory_embeddings" / "banking_policy_db"
        )
        db_path.mkdir(parents=True, exist_ok=True)

        self.embedding_function = SentenceTransformerEmbeddingFunction(
            model_name=embedding_model
        )
        self.client = chromadb.PersistentClient(path=str(db_path))
        self.collection = self.client.get_or_create_collection(
            name=collection_name,
            embedding_function=self.embedding_function,
            metadata={"description": "Mock banking policy knowledge base"},
        )
        self._seed_collection()

    def _seed_collection(self) -> None:
        """Add mock policies only once; Chroma ignores already-known IDs safely."""
        existing = self.collection.count()
        if existing > 0:
            return

        self.collection.add(
            ids=list(POLICY_DOCS.keys()),
            documents=list(POLICY_DOCS.values()),
            metadatas=[{"source": "mock_banking_policy", "policy_id": key} for key in POLICY_DOCS],
        )

    def retrieve_with_metadata(self, query: str, k: int = 3) -> list[dict[str, Any]]:
        """Return top-k semantically retrieved policy chunks and their metadata."""
        results = self.collection.query(
            query_texts=[query],
            n_results=min(k, max(1, self.collection.count())),
            include=["documents", "metadatas", "distances"],
        )

        documents = results.get("documents", [[]])[0] or []
        metadatas = results.get("metadatas", [[]])[0] or []
        distances = results.get("distances", [[]])[0] or []
        return [
            {
                "text": document,
                "metadata": metadata or {},
                "distance": round(float(distance), 4),
            }
            for document, metadata, distance in zip(documents, metadatas, distances)
        ]

    def retrieve(self, query: str, k: int = 3) -> str | None:
        """Compatibility method used by the agent; returns joined RAG context."""
        chunks = self.retrieve_with_metadata(query, k=k)
        if not chunks:
            return None
        return "\n\n".join(
            f"[{chunk['metadata'].get('policy_id', 'policy')}] {chunk['text']}"
            for chunk in chunks
        )

In [5]:
if __name__ == "__main__":
    rag = BankingRAG(persist_directory = PROJECT_ROOT)
    for item in rag.retrieve_with_metadata("What is the Gold Card annual fee?"):
        print(item)

C:\Users\HP\anaconda3\envs\agentic_workflows\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 1153.98it/s]


{'text': 'Gold Credit Card: Annual Fee is $50. The annual fee is waived if annual card spending exceeds $10,000. Interest rate is 18% APR.', 'metadata': {'source': 'mock_banking_policy', 'policy_id': 'gold_card_fee'}, 'distance': 0.1643}
{'text': 'Student Account: Zero minimum balance is required. There are no monthly maintenance fees. Maximum daily cash withdrawal is $500.', 'metadata': {'source': 'mock_banking_policy', 'policy_id': 'student_account'}, 'distance': 0.6194}
{'text': 'Personal Loan: Applicants generally require a credit score above 700 and annual income above $50,000. Maximum loan tenure is five years.', 'metadata': {'policy_id': 'loan_eligibility', 'source': 'mock_banking_policy'}, 'distance': 0.8095}
